# 🎭 Sentiment Analyzer — ML Project

**Goal:** Build a binary text classifier that labels sentences as *Positive* or *Negative* using a **TF-IDF + Logistic Regression** pipeline.

| Step | What we do |
|------|------------|
| 1 | Explore the dataset |
| 2 | Train the model |
| 3 | Evaluate with metrics & plots |
| 4 | Run custom predictions |

## 1 · Imports & Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')   # required for notebook compatibility in some envs
import matplotlib.pyplot as plt
import seaborn as sns

from data  import load_dataset, get_splits
from model import (
    build_pipeline, train, load_model, predict,
    plot_confusion_matrix, plot_roc_curve,
    plot_top_features, plot_score_distribution,
    MODEL_PATH, PLOTS_DIR,
)

%matplotlib inline
print('✅ All imports successful')

## 2 · Explore the Dataset

In [ ]:
df = load_dataset()
print(f'Shape  : {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(8)

In [ ]:
# Class distribution
counts = df['label'].value_counts().rename({0: 'Negative', 1: 'Positive'})

fig, ax = plt.subplots(figsize=(4, 3))
ax.bar(counts.index, counts.values,
       color=['#E05C5C', '#4CAF82'], edgecolor='white', width=0.5)
ax.set_title('Class Distribution', fontweight='bold')
ax.set_ylabel('Count')
for i, v in enumerate(counts.values):
    ax.text(i, v + 0.5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Text length analysis
df['word_count'] = df['text'].str.split().str.len()
df['char_count'] = df['text'].str.len()
df.groupby('label')[['word_count', 'char_count']].agg(['mean', 'min', 'max']).round(1)

## 3 · Train the Model

**Pipeline:**
```
Raw text  →  TF-IDF (unigrams + bigrams, 5 000 features)  →  Logistic Regression
```

In [ ]:
model, metrics = train(save=True)

## 4 · Evaluate the Model

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────
fig = plot_confusion_matrix(metrics['cm'], save=False)
plt.show()

In [ ]:
# ── ROC Curve ────────────────────────────────────────────────────────────
fig = plot_roc_curve(metrics['fpr'], metrics['tpr'], metrics['roc_auc'], save=False)
plt.show()

In [ ]:
# ── Top features ─────────────────────────────────────────────────────────
fig = plot_top_features(model, n=15, save=False)
plt.show()

In [ ]:
# ── Score distribution ───────────────────────────────────────────────────
fig = plot_score_distribution(model, save=False)
plt.show()

## 5 · Metrics Summary

In [ ]:
from sklearn.metrics import classification_report

print(f"Accuracy : {metrics['accuracy']:.4f}")
print(f"ROC-AUC  : {metrics['roc_auc']:.4f}")
print()
print(classification_report(
    metrics['y_test'], metrics['y_pred'],
    target_names=['Negative', 'Positive']
))

## 6 · Make Custom Predictions

Add your own sentences in the list below and run the cell!

In [ ]:
my_texts = [
    "I absolutely loved this, it was fantastic!",
    "Terrible quality, broke after one day.",
    "It was okay, nothing special.",
    "Best experience of my life, highly recommend!",
    "Very disappointing and a waste of money.",
    # ← add more sentences here
]

results = predict(my_texts, model)
results

In [ ]:
# Visual confidence bars
fig, ax = plt.subplots(figsize=(8, max(3, len(results) * 0.7)))

colors = ['#4CAF82' if s == 'Positive' else '#E05C5C'
          for s in results['sentiment']]
labels = [f"{row.sentiment} ({row.confidence})" for _, row in results.iterrows()]
short_texts = [t[:55] + '…' if len(t) > 55 else t for t in results['text']]

bars = ax.barh(short_texts, results['pos_score'], color=colors, alpha=0.85,
               edgecolor='white')
ax.axvline(0.5, color='#555', linestyle='--', linewidth=1.2,
           label='Decision boundary')
ax.set_xlim(0, 1)
ax.set_xlabel('Positive-class probability')
ax.set_title('Prediction Confidence', fontweight='bold')
ax.bar_label(bars, labels=labels, padding=4, fontsize=9)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 7 · Experiment: Hyperparameter Effect

Try different values of the regularisation parameter **C** and see how accuracy changes.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = get_splits()
C_values = [0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0]

train_accs, test_accs = [], []
for C in C_values:
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=5000,
                                  sublinear_tf=True, stop_words='english')),
        ('clf',   LogisticRegression(C=C, max_iter=1000, solver='lbfgs')),
    ])
    pipe.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, pipe.predict(X_train)))
    test_accs.append(accuracy_score(y_test,  pipe.predict(X_test)))

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogx(C_values, train_accs, 'o-', color='#4CAF82', label='Train accuracy')
ax.semilogx(C_values, test_accs,  's--', color='#E05C5C', label='Test accuracy')
ax.set_xlabel('C  (regularisation strength, log scale)')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy vs Regularisation (C)', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best_C = C_values[np.argmax(test_accs)]
print(f'Best C on test set: {best_C}  (accuracy = {max(test_accs):.4f})')

---
### 🏁 Project Summary

| Component | Choice |
|-----------|--------|
| Vectoriser | TF-IDF (1-gram + 2-gram, top 5 000 features) |
| Classifier | Logistic Regression (C = 1.0) |
| Dataset | 200 hand-labeled sentences (balanced) |
| Evaluation | Accuracy, Precision, Recall, F1, ROC-AUC |

**Ideas to improve:**
- Add more training data (or use a public dataset like IMDb / SST-2)
- Try `LinearSVC`, `RandomForest`, or gradient-boosted trees
- Use word embeddings (Word2Vec, GloVe) instead of TF-IDF
- Fine-tune a pre-trained transformer (BERT, DistilBERT) for much higher accuracy